# recount2 — C2CP Pathway Coverage vs Sample Size (multiplier=100, max.iter=1000, max.U.updates=20)

**Environment:** `clamp-analyses`

Plots the C2CP pathway coverage and K results produced by `09_recount2_coverage_multiplier_imp_fixed_maxU20.ipynb`.

Coverage = proportion of input C2CP pathways significantly associated with an LV (FDR < 0.05),
following the `GetPathwayCoverage` approach from the multi-plier paper (Taroni 2018).

Input: `output/recount2_multiplier_imp_fixed_maxU20/c2cp_subsample_{N}_seed_{idx}/CLAMPfull_C2CP.rds`

## Libraries

In [ ]:
library(dplyr)
library(ggplot2)
library(here)

source(here("config.R"))

## Configuration

In [ ]:
FDR_CUTOFF   <- 0.05
output_dir   <- file.path(config$GENERAL$OUTPUT_DIR, "recount2_multiplier_imp_fixed_maxU20")
sample_sizes <- c(500, 1000, 2000, 4000, 8000, 16000, 32000)
n_seeds      <- 3

# Load CLAMP_K values from the results summary produced by 09_ (if available)
summary_csv <- file.path(output_dir, "subsample_results_summary_multiplier_imp_fixed_maxU20.csv")
if (file.exists(summary_csv)) {
    results_summary <- read.csv(summary_csv, stringsAsFactors = FALSE)
    message("Loaded results summary from: ", summary_csv)
} else {
    results_summary <- NULL
    message("Results summary not found — will read CLAMP_K.rds per run as fallback")
}

## Coverage function

In [ ]:
# Adapted from GetPathwayCoverage (Taroni 2018 / multi-plier)
GetPathwayCoverage <- function(clamp.result, fdr.cutoff = 0.05) {
    summary.df     <- clamp.result$summary
    input.pathways <- colnames(clamp.result$C)
    num.lvs        <- ncol(clamp.result$U)

    sig.pathways <- unique(summary.df$pathway[which(summary.df$FDR < fdr.cutoff)])
    sig.lvs      <- unique(summary.df$LV[which(summary.df$FDR < fdr.cutoff)])

    list(
        pathway           = length(sig.pathways) / length(input.pathways),
        lv                = length(sig.lvs) / num.lvs,
        sig.pathway.by.lv = length(sig.pathways) / num.lvs
    )
}

## Collect results per run

In [ ]:
rows <- list()

for (n_target in sample_sizes) {
    for (run_idx in seq_len(n_seeds)) {
        run_dir  <- file.path(output_dir,
                              paste0("c2cp_subsample_", n_target, "_seed_", run_idx))
        rds_path <- file.path(run_dir, "CLAMPfull_C2CP.rds")

        if (!file.exists(rds_path)) {
            message("Not found — skipping: c2cp_subsample_",
                    n_target, "_seed_", run_idx)
            next
        }

        # Look up CLAMP_K from summary table, fall back to per-run .rds
        if (!is.null(results_summary)) {
            row_match <- results_summary[
                results_summary$sample_size == n_target &
                results_summary$run         == run_idx, ]
            clamp_k <- if (nrow(row_match) == 1) row_match$CLAMP_K else NA_integer_
        } else {
            k_path  <- file.path(run_dir, "CLAMP_K.rds")
            clamp_k <- if (file.exists(k_path)) readRDS(k_path) else NA_integer_
        }

        model <- readRDS(rds_path)
        cov   <- GetPathwayCoverage(model, fdr.cutoff = FDR_CUTOFF)

        rows[[length(rows) + 1]] <- data.frame(
            sample_size      = n_target,
            n_samples        = ncol(model$B),
            run              = run_idx,
            pathway_coverage = cov$pathway,
            lv_coverage      = cov$lv,
            clamp_k          = clamp_k
        )
        message(sprintf("n=%6d  run=%d  pathway_cov=%.2f%%  K=%d",
                        n_target, run_idx,
                        cov$pathway * 100, clamp_k))
    }
}

results_df <- dplyr::bind_rows(rows) %>%
    dplyr::mutate(sample_size = factor(sample_size,
                                       levels = sort(unique(sample_size))))
print(results_df)

## Pathway coverage

In [ ]:
p_cov <- ggplot(results_df,
                aes(x = sample_size, y = pathway_coverage * 100)) +
    geom_boxplot(fill = "#FF9800", color = "#E65100",
                 alpha = 0.6, outlier.shape = NA, width = 0.5) +
    geom_jitter(color = "#E65100", width = 0.1, size = 2.5, alpha = 0.8) +
    labs(
        x       = "Number of samples",
        y       = "Pathway coverage (%)",
        caption = paste0("FDR < ", FDR_CUTOFF,
                         "; multiplier=100; max.iter=1000; max.U.updates=20; 3 seeds per sample size")
    ) +
    theme_bw(base_size = 13) +
    theme(
        axis.text.x      = element_text(angle = 45, hjust = 1),
        panel.grid.minor = element_blank()
    )

print(p_cov)

## K (number of components)

In [ ]:
p_k <- ggplot(results_df,
              aes(x = sample_size, y = clamp_k)) +
    geom_boxplot(fill = "#FF9800", color = "#E65100",
                 alpha = 0.6, outlier.shape = NA, width = 0.5) +
    geom_jitter(color = "#E65100", width = 0.1, size = 2.5, alpha = 0.8) +
    labs(
        x = "Number of samples",
        y = "K (number of components)"
    ) +
    theme_bw(base_size = 13) +
    theme(
        axis.text.x      = element_text(angle = 45, hjust = 1),
        panel.grid.minor = element_blank()
    )

print(p_k)

## Summary table

In [ ]:
results_df %>%
    dplyr::group_by(sample_size) %>%
    dplyr::summarise(
        median_cov = round(median(pathway_coverage) * 100, 2),
        min_cov    = round(min(pathway_coverage) * 100, 2),
        max_cov    = round(max(pathway_coverage) * 100, 2),
        median_k   = median(clamp_k, na.rm = TRUE),
        n          = dplyr::n(),
        .groups    = "drop"
    ) %>%
    dplyr::arrange(sample_size)